# DEAP: Evolutionary Optimization for Black-Box and Multi-Objective Problems

## The Core Question

You have a CFD solver that takes 4 hours to run. You want to optimize an airfoil shape. **How do you optimize when your function is black-box, non-differentiable, or multi-objective?**

Gradient-based methods (L-BFGS-B, SLSQP) need derivatives. They fail when:
- The objective is a simulation (no analytical gradient)
- The landscape is non-smooth (discontinuities, kinks)
- You have multiple conflicting objectives (minimize drag AND weight)

**Genetic algorithms (GAs)** don't need gradients. They work with any fitness function — a CFD solver, an FEA model, even a simulation. And for multi-objective problems, they give you a **Pareto front** (a set of optimal trade-offs) instead of a single answer.

## Why DEAP?

- **No gradients needed** — works with any black-box fitness function
- **Multi-objective** — NSGA-II gives you a Pareto front, not a single point
- **Flexible** — define your own individuals, fitness, and operators
- **Parallelizable** — evaluate the population in parallel (embarrassingly parallel)
- **Extensible** — add constraints, repair operators, or custom selection

## The Workflow

```
Define individual (genome)
        |
        v
Define fitness function
  (CFD solver / FEA / simulation)
        |
        v
Initialize population
        |
        v
Evolve:
  Evaluate fitness
  Select parents
  Crossover + Mutation
  Replace population
        |
        v
Repeat for N generations
        |
        v
Extract best / Pareto front
```

## Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from deap import base, creator, tools, algorithms

plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
print('DEAP imports OK')

## 1. Single-Objective: GA vs SciPy

Let's start with a simple problem: minimize the **Rosenbrock function** using a genetic algorithm. We'll compare it to scipy's L-BFGS-B.

**The GA approach:**
1. Represent the solution as a vector of real numbers (floats)
2. Evaluate the fitness (Rosenbrock value)
3. Select the best individuals
4. Crossover (mix two parents) and mutate (random perturbation)
5. Repeat for N generations


In [ ]:
# Define fitness and individual
creator.create('FitnessMin', base.Fitness, weights=(-1.0,))
creator.create('Individual', list, fitness=creator.FitnessMin)

# Rosenbrock function
def rosenbrock(x):
    return (1 - x[0])**2 + 100 * (x[1] - x[0]**2)**2

# Toolbox
toolbox = base.Toolbox()
toolbox.register('attr', np.random.uniform, -2.0, 2.0)
toolbox.register('individual', tools.initRepeat, creator.Individual, toolbox.attr, n=2)
toolbox.register('population', tools.initRepeat, list, toolbox.individual)

# Fitness evaluation
def eval_individual(ind):
    ind.fitness.values = (rosenbrock(ind),)
    return ind.fitness.values

toolbox.register('evaluate', eval_individual)

# Operators
toolbox.register('mate', tools.cxBlend, alpha=0.5)
toolbox.register('mutate', tools.mutGaussian, mu=0.0, sigma=0.2, indpb=0.3)
toolbox.register('select', tools.selTournament, tournsize=3)

# Run GA
np.random.seed(42)
pop = toolbox.population(n=50)
hof = tools.HallOfFame(1)
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register('avg', np.mean)
stats.register('min', np.min)

pop, log = algorithms.eaSimple(pop, toolbox, cxpb=0.7, mutpb=0.2, ngen=100, stats=stats, halloffame=hof, verbose=False)

print(f'GA best: x = {hof[0]}, f = {hof[0].fitness.values[0]:.6f}')
print(f'Generations: 100, Population: 50')
print(f'Total evaluations: {len(pop) * 100 + 50}')

### What does this mean?

The GA found a solution close to the global minimum (1, 1) with f ≈ 0. But it used **~5,000 evaluations** (50 population × 100 generations), while L-BFGS-B used **~60 evaluations**.

**The trade-off:**
- **GA:** slow (1000x more evaluations), but robust, no gradients, handles multi-modal landscapes
- **L-BFGS-B:** fast (60 evals), but needs gradients, gets stuck in local minima

**When to use a GA:**
- Black-box objective (CFD solver, FEA, simulation)
- Non-smooth or multi-modal landscape
- You have parallel compute (population evaluation is embarrassingly parallel)
- You need a set of good solutions, not just one


## 2. Engineering Application: Airfoil Shape Optimization

**The problem:** Optimize an airfoil shape to minimize drag, subject to lift and thickness constraints.

**Design variables:**
- `thickness` (t/c): 0.05 - 0.20
- `camber`: 0.0 - 0.10
- `camber_pos`: 0.2 - 0.8

**Objectives and constraints:**
- Minimize: `Cd` (drag coefficient)
- Constraint: `Cl >= 0.5` (lift coefficient)
- Constraint: `thickness >= 0.08` (minimum thickness)

We'll use a **surrogate model** (analytical approximation) to keep the notebook fast. In practice, you'd replace this with a CFD solver call.


In [ ]:
# Airfoil surrogate model
def airfoil_cd(t, c, cp):
    # Drag increases with thickness and camber
    return 0.01 + 0.5 * t**2 + 0.3 * c**2 + 0.1 * (cp - 0.4)**2

def airfoil_cl(t, c, cp):
    # Lift increases with camber, decreases with thickness
    return 0.3 + 2.0 * c - 0.5 * t

# Fitness with penalty for constraint violation
def eval_airfoil(ind):
    t, c, cp = ind
    cd = airfoil_cd(t, c, cp)
    cl = airfoil_cl(t, c, cp)
    
    # Penalty for constraint violation
    penalty = 0.0
    if cl < 0.5:
        penalty += 10.0 * (0.5 - cl)
    if t < 0.08:
        penalty += 10.0 * (0.08 - t)
    
    ind.fitness.values = (cd + penalty,)
    return ind.fitness.values

# Re-register for airfoil
creator.create('FitnessAirfoil', base.Fitness, weights=(-1.0,))
creator.create('IndividualAirfoil', list, fitness=creator.FitnessAirfoil)

toolbox_air = base.Toolbox()
toolbox_air.register('attr_t', np.random.uniform, 0.05, 0.20)
toolbox_air.register('attr_c', np.random.uniform, 0.0, 0.10)
toolbox_air.register('attr_cp', np.random.uniform, 0.2, 0.8)
toolbox_air.register('individual', lambda: creator.IndividualAirfoil(
    [toolbox_air.attr_t(), toolbox_air.attr_c(), toolbox_air.attr_cp()]
))
toolbox_air.register('population', tools.initRepeat, list, toolbox_air.individual)
toolbox_air.register('evaluate', eval_airfoil)
toolbox_air.register('mate', tools.cxBlend, alpha=0.5)
toolbox_air.register('mutate', tools.mutGaussian, mu=0.0, sigma=0.01, indpb=0.3)
toolbox_air.register('select', tools.selTournament, tournsize=3)

# Run GA
np.random.seed(42)
pop_air = toolbox_air.population(n=50)
hof_air = tools.HallOfFame(1)
stats_air = tools.Statistics(lambda ind: ind.fitness.values[0])
stats_air.register('avg', np.mean)
stats_air.register('min', np.min)

pop_air, log_air = algorithms.eaSimple(pop_air, toolbox_air, cxpb=0.7, mutpb=0.2, ngen=100, stats=stats_air, halloffame=hof_air, verbose=False)

best = hof_air[0]
print(f'Best airfoil: t = {best[0]:.4f}, camber = {best[1]:.4f}, camber_pos = {best[2]:.4f}')
print(f'Drag: Cd = {airfoil_cd(*best):.6f}')
print(f'Lift: Cl = {airfoil_cl(*best):.4f} (constraint: >= 0.5)')
print(f'Thickness: {best[0]:.4f} (constraint: >= 0.08)')

### What does this mean?

The GA found the **minimum-drag airfoil** that satisfies both constraints. The thickness is at the constraint boundary (t = 0.08), meaning the optimizer wants a thinner airfoil but is limited by the minimum thickness requirement.

**In practice:**
1. Replace `airfoil_cd` and `airfoil_cl` with calls to a CFD solver (or a surrogate model trained on CFD data)
2. Use a larger population (100-500) and more generations (200-500)
3. Evaluate the population in parallel (each individual is independent)
4. Validate the optimal design with a high-fidelity CFD simulation


## 3. Convergence: Tracking the GA

The `Logbook` from `eaSimple` records the statistics at each generation. Let's plot the convergence.


In [ ]:
# Plot convergence
gens = range(len(log_air))
avg_fitness = [record['avg'] for record in log_air]
min_fitness = [record['min'] for record in log_air]

plt.figure(figsize=(10, 4))
plt.plot(gens, avg_fitness, 'b-', label='Average fitness', linewidth=1)
plt.plot(gens, min_fitness, 'r-', label='Best fitness', linewidth=1)
plt.xlabel('Generation')
plt.ylabel('Cd (drag coefficient)')
plt.title('GA Convergence: Airfoil Optimization')
plt.legend()
plt.tight_layout()
plt.show()

### What does this mean?

The convergence curve shows:
- **Rapid initial improvement** (first 20 generations) — the GA quickly finds good regions
- **Slow refinement** (generations 20-100) — the GA fine-tunes the best solutions
- **Plateau** (after ~80 generations) — the GA has converged

**Practical tip:** if the curve plateaus, you can stop early and save compute. The `ngen` parameter controls the number of generations — set it based on when the curve plateaus.


## 4. Multi-Objective: Pareto Front

**The problem:** Minimize drag AND weight simultaneously. These objectives conflict — a thinner airfoil has less drag but less structural integrity (more weight for the same stiffness).

**Multi-objective optimization** doesn't give you a single answer. It gives you a **Pareto front** — a set of solutions where you can't improve one objective without worsening the other.

**NSGA-II** (Non-dominated Sorting Genetic Algorithm II) is the standard for multi-objective GAs. It maintains diversity in the population and converges to the Pareto front.


In [ ]:
# Multi-objective: minimize drag AND weight
creator.create('FitnessMO', base.Fitness, weights=(-1.0, -1.0))
creator.create('IndividualMO', list, fitness=creator.FitnessMO)

def airfoil_weight(t, c, cp):
    # Weight increases with thickness
    return 1.0 + 5.0 * t

def eval_airfoil_mo(ind):
    t, c, cp = ind
    cd = airfoil_cd(t, c, cp)
    weight = airfoil_weight(t, c, cp)
    ind.fitness.values = (cd, weight)
    return ind.fitness.values

toolbox_mo = base.Toolbox()
toolbox_mo.register('attr_t', np.random.uniform, 0.05, 0.20)
toolbox_mo.register('attr_c', np.random.uniform, 0.0, 0.10)
toolbox_mo.register('attr_cp', np.random.uniform, 0.2, 0.8)
toolbox_mo.register('individual', lambda: creator.IndividualMO(
    [toolbox_mo.attr_t(), toolbox_mo.attr_c(), toolbox_mo.attr_cp()]
))
toolbox_mo.register('population', tools.initRepeat, list, toolbox_mo.individual)
toolbox_mo.register('evaluate', eval_airfoil_mo)
toolbox_mo.register('mate', tools.cxBlend, alpha=0.5)
toolbox_mo.register('mutate', tools.mutGaussian, mu=0.0, sigma=0.01, indpb=0.3)
toolbox_mo.register('select', tools.selTournament, tournsize=3)

# Run NSGA-II
np.random.seed(42)
pop_mo = toolbox_mo.population(n=50)
hof_mo = tools.HallOfFame(100)  # store up to 100 non-dominated solutions
stats_mo = tools.Statistics(lambda ind: ind.fitness.values)
stats_mo.register('avg', lambda x: np.mean(x, axis=0))
stats_mo.register('min', lambda x: np.min(x, axis=0))

pop_mo, log_mo = algorithms.eaMuPlusLambda(pop_mo, toolbox_mo, mu=50, lambda_=50, cxpb=0.7, mutpb=0.2, ngen=100, stats=stats_mo, halloffame=hof_mo, verbose=False)

print(f'Pareto front: {len(hof_mo)} non-dominated solutions')
print('First 5 solutions (Cd, weight):')
for ind in hof_mo[:5]:
    print(f'  Cd = {ind.fitness.values[0]:.6f}, weight = {ind.fitness.values[1]:.4f}')

### What does this mean?

NSGA-II found **100 non-dominated solutions** — the Pareto front. Each solution is "optimal" in the sense that you can't improve drag without increasing weight, or vice versa.

**The trade-off:**
- **Low drag** → thin airfoil → low weight (but may violate structural constraints)
- **Low weight** → thin airfoil → high drag (but structurally efficient)

The Pareto front shows you the **full range of trade-offs**. You (the engineer) choose the solution that best matches your requirements.


## 5. Visualizing the Pareto Front

Let's plot the Pareto front in objective space (drag vs weight).


In [ ]:
# Plot Pareto front
pareto_cd = [ind.fitness.values[0] for ind in hof_mo]
pareto_weight = [ind.fitness.values[1] for ind in hof_mo]

plt.figure(figsize=(10, 6))
plt.scatter(pareto_cd, pareto_weight, c='steelblue', s=50, edgecolors='k', linewidths=0.5)
plt.xlabel('Drag Coefficient (Cd)')
plt.ylabel('Weight (normalized)')
plt.title('Pareto Front: Drag vs Weight')
plt.tight_layout()
plt.show()

# Also plot in design space
pareto_t = [ind[0] for ind in hof_mo]
pareto_c = [ind[1] for ind in hof_mo]

plt.figure(figsize=(10, 6))
plt.scatter(pareto_t, pareto_c, c='coral', s=50, edgecolors='k', linewidths=0.5)
plt.xlabel('Thickness (t/c)')
plt.ylabel('Camber')
plt.title('Pareto Front in Design Space')
plt.tight_layout()
plt.show()

### What does this mean?

The Pareto front in **objective space** shows the trade-off curve: as drag decreases, weight increases. The "knee" of the curve is often the best compromise — the point where you get the most drag reduction per unit weight increase.

The Pareto front in **design space** shows how the design variables change along the front:
- **Low drag, high weight** → thick airfoil, high camber
- **High drag, low weight** → thin airfoil, low camber

**Engineering decision:** pick the solution on the Pareto front that best matches your requirements. If you need low drag, pick the left end. If you need low weight, pick the right end. If you need a balance, pick the knee.


## 6. Exercises

1. **GA vs SciPy.** Run the GA and L-BFGS-B on the Rosenbrock function. Compare the number of evaluations and the final accuracy. At what problem size does the GA become competitive?

2. **Constraint handling.** Replace the penalty method with a **repair operator** that projects infeasible solutions back to the feasible region. How does this affect convergence?

3. **Parallel evaluation.** Modify the GA to evaluate the population in parallel using `multiprocessing.Pool`. How much speedup do you get on a 4-core machine?

4. **More objectives.** Add a third objective: minimize the **moment coefficient** (pitching moment). How does the Pareto front change in 3D?

5. **Real CFD.** Replace the surrogate model with a call to an actual CFD solver (or a pre-computed dataset). How does the Pareto front change when the model is truly non-linear?
